In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client  = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

print(response.output_text)

Absolutely — in most cases, yes, you can join if the course is still open or if the instructor allows late enrollment.

To check, look for:
- the registration/enrollment deadline
- whether the course has available seats
- any prerequisites or approval requirements
- whether late add/drop is still permitted

If you want, I can help you draft a short message to the instructor or registrar asking to join.


In [4]:
def search(query):
    boost_dict  = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
    )

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [6]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join late enrollment registration"}', call_id='call_01EZMmEqp8T2sogwE8fN2ocb', name='search', type='function_call', id='fc_02af56b098f8fc83006a64e3db95f081a0ab1d534b77383ba7', caller=None, namespace=None, status='completed')]

In [7]:
import json

In [8]:
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join late enrollment registration"}', call_id='call_01EZMmEqp8T2sogwE8fN2ocb', name='search', type='function_call', id='fc_02af56b098f8fc83006a64e3db95f081a0ab1d534b77383ba7', caller=None, namespace=None, status='completed')

In [9]:
args = json.loads(call.arguments)
args

{'query': 'join course discovered course can I join late enrollment registration'}

In [10]:
results = search(**args)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [11]:
results_json = json.dumps(results, indent=2)
results_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",\n    "answer": "You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the

In [12]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [13]:
messages.extend(response.output)

messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join late enrollment registration"}', call_id='call_01EZMmEqp8T2sogwE8fN2ocb', name='search', type='function_call', id='fc_02af56b098f8fc83006a64e3db95f081a0ab1d534b77383ba7', caller=None, namespace=None, status='completed')]

In [14]:
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": results_json,
})

messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join late enrollment registration"}', call_id='call_01EZMmEqp8T2sogwE8fN2ocb', name='search', type='function_call', id='fc_02af56b098f8fc83006a64e3db95f081a0ab1d534b77383ba7', caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_01EZMmEqp8T2sogwE8fN2ocb',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can 

In [15]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes — you can still join and start learning.

If you want a certificate, the key thing is to submit your project while the course is still accepting submissions.


In [16]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(772, 36)

In [17]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15 / 1_000_000
    OUTPUT_PRICE_PER_MILLION = 0.60 / 1_000_000

    input_cost = (input_tokens) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [18]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [19]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [23]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]


In [24]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [25]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment registration late join FAQ"}
function_call: search {"query":"new student join course can I still enroll discovered the course FAQ"}


In [21]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [22]:
print(response.output_text)

Yes — you can still join the course.

You can start learning and submitting homework as long as the submission form is open. If you want a certificate, you’ll need to submit your project while the course is still accepting submissions.

Would you like me to help with anything else about the course?


In [ ]:
iteration = 1

while True:
    print(f"Iteration #{iteration}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    iteration += 1
    if has_function_calls == False:
        break

    

Iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, you need to submit your project while the course is still accepting submissions.

Would you like to know more about registration, homework, or certificates?


In [28]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    iteration = 1

    while True:
        print(f"Iteration #{iteration}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        iteration = iteration + 1
        if has_function_calls == False:
            break

    return last_answer

In [29]:
agent_loop(instructions, "How do I run Olama locally?")

Iteration #1...
function_call: search {"query":"Olama local run install Ollama locally run model command line start server FAQ"}
Iteration #2...
function_call: search {"query":"ollama serve localhost 11434 connection refused run locally python client FAQ"}
Iteration #3...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - macOS: download and install from https://ollama.com/download
   - Windows: download the `.msi` from the same page
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This downloads the model, starts it locally, and opens a chat prompt.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response with model info.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "use

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - macOS: download and install from https://ollama.com/download\n   - Windows: download the `.msi` from the same page\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model, starts it locally, and opens a chat prompt.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response with model info.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server:\n```bash\nnohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you want, I can also show you how to 

In [32]:
result = agent_loop(instructions, "I just discovered the course. Can I still join it?")

Iteration #1...


function_call: search {"query":"join course late discovered course still join enrollment start late FAQ"}
Iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, the key thing is to submit your project while submissions are still being accepted. The course materials are available, so you can start whenever you want.

If you'd like, I can also help you figure out the best way to catch up quickly. Are there other areas you want to explore?


In [33]:
print(result)

Yes — you can still join the course.

If you want a certificate, the key thing is to submit your project while submissions are still being accepted. The course materials are available, so you can start whenever you want.

If you'd like, I can also help you figure out the best way to catch up quickly. Are there other areas you want to explore?


In [34]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()


agent_loop(instructions, "I just discovered the course. Can I join it?")

Iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment registration open late join after start FAQ"}
Iteration #2...
function_call: search {"query":"certificate project accepting submissions while accepting submissions late join project certificate FAQ llm zoomcamp"}
Iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. You can also start learning and working through the materials even if you joined late.

If you want, I can also explain how the course workflow works or what’s needed for the certificate.


'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open. You can also start learning and working through the materials even if you joined late.\n\nIf you want, I can also explain how the course workflow works or what’s needed for the certificate.'

In [35]:
agent_loop(instructions, "what's queen gambit?")

Iteration #1...
function_call: search {"query":"queen gambit chess opening queen gambit"}
Iteration #2...
function_call: search {"query":"queen's gambit chess opening definition"}
Iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. `d4 d5`
2. `c4`

White offers the c-pawn to try to distract Black’s `d5` pawn and gain better control of the center. It’s one of the most famous and classical chess openings.

A common response by Black is:
- **Queen’s Gambit Accepted**: `1.d4 d5 2.c4 dxc4`
- **Queen’s Gambit Declined**: `1.d4 d5 2.c4 e6` or `...c6`

If you want, I can also explain:
- the idea behind the opening,
- the difference between accepted and declined,
- or show a simple example line.


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. `d4 d5`\n2. `c4`\n\nWhite offers the c-pawn to try to distract Black’s `d5` pawn and gain better control of the center. It’s one of the most famous and classical chess openings.\n\nA common response by Black is:\n- **Queen’s Gambit Accepted**: `1.d4 d5 2.c4 dxc4`\n- **Queen’s Gambit Declined**: `1.d4 d5 2.c4 e6` or `...c6`\n\nIf you want, I can also explain:\n- the idea behind the opening,\n- the difference between accepted and declined,\n- or show a simple example line.'

In [36]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


agent_loop(instructions, "what's queen gambit?")

Iteration #1...
function_call: search {"query":"queen gambit"}
Iteration #2...
function_call: search {"query":"queen's gambit chess opening course FAQ"}
Iteration #3...
ASSISTANT:
I couldn’t find anything in the course FAQ about “Queen’s Gambit,” so it looks like that’s not a course-related topic.

If you meant something else in the course context, feel free to rephrase it, and I can check again. Are there other areas you want to explore?


'I couldn’t find anything in the course FAQ about “Queen’s Gambit,” so it looks like that’s not a course-related topic.\n\nIf you meant something else in the course context, feel free to rephrase it, and I can check again. Are there other areas you want to explore?'